# Intent Dataset EDA 

In [1]:
import os
import re
import numpy as np
import pandas as pd

TRAIN_PATH = "intent_analysis_train.csv"
VAL_PATH = "intent_analysis_val.csv"
TEST_PATH = "intent_analysis_test.csv"

def normalize_query(text):
    text = str(text or "").strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def load_df(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    if "query" not in df.columns or "intent" not in df.columns:
        raise ValueError(f"Missing columns in {path}")
    df["query"] = df["query"].astype(str)
    df["intent"] = df["intent"].astype(str)
    df["nq"] = df["query"].map(normalize_query)
    return df

train_df = load_df(TRAIN_PATH)
val_df = load_df(VAL_PATH)
test_df = load_df(TEST_PATH)

datasets = {"train": train_df, "val": val_df, "test": test_df}
print({k: len(v) for k, v in datasets.items()})

{'train': 2806, 'val': 312, 'test': 225}


In [2]:
def basic_counts(df):
    total = len(df)
    unique = df["nq"].nunique(dropna=True)
    dupes = total - unique
    missing_intent = df["intent"].isna().sum()
    return {"total": total, "unique": unique, "dupes": dupes, "missing_intent": missing_intent}

for name, df in datasets.items():
    print("\n==", name, "==")
    print(basic_counts(df))
    dist = df["intent"].value_counts().rename("count").to_frame()
    dist["pct"] = (dist["count"] / len(df)).round(4)
    print(dist)


== train ==
{'total': 2806, 'unique': 2806, 'dupes': 0, 'missing_intent': np.int64(0)}
               count     pct
intent                      
color_variant   1006  0.3585
graph_pairing    917  0.3268
similar_items    883  0.3147

== val ==
{'total': 312, 'unique': 312, 'dupes': 0, 'missing_intent': np.int64(0)}
               count     pct
intent                      
color_variant    112  0.3590
graph_pairing    102  0.3269
similar_items     98  0.3141

== test ==
{'total': 225, 'unique': 225, 'dupes': 0, 'missing_intent': np.int64(0)}
               count     pct
intent                      
graph_pairing     76  0.3378
color_variant     75  0.3333
similar_items     74  0.3289


In [3]:
def add_lengths(df):
    df = df.copy()
    df["len_chars"] = df["query"].str.len()
    df["len_tokens"] = df["query"].str.split().map(lambda x: len(x) if isinstance(x, list) else 0)
    return df

for name, df in datasets.items():
    tmp = add_lengths(df)
    print("\n==", name, "length stats==")
    print(tmp[["len_chars", "len_tokens"]].describe().round(2))


== train length stats==
       len_chars  len_tokens
count    2806.00     2806.00
mean       40.16        7.68
std        23.53        3.98
min         9.00        3.00
25%        22.00        5.00
50%        29.00        6.00
75%        64.00       11.00
max       111.00       19.00

== val length stats==
       len_chars  len_tokens
count     312.00      312.00
mean       39.67        7.66
std        23.18        3.98
min        10.00        3.00
25%        22.00        5.00
50%        29.00        6.00
75%        64.00       11.00
max        92.00       17.00

== test length stats==
       len_chars  len_tokens
count     225.00      225.00
mean       35.84        6.63
std         8.49        1.46
min        17.00        4.00
25%        30.00        6.00
50%        36.00        7.00
75%        41.00        7.00
max        63.00       11.00


In [6]:
train_set = set(train_df["nq"].dropna())
val_set = set(val_df["nq"].dropna())
test_set = set(test_df["nq"].dropna())

overlap_train_val = len(train_set & val_set)
overlap_train_test = len(train_set & test_set)
overlap_val_test = len(val_set & test_set)
overlap_all = len(train_set & val_set & test_set)

print({
    "train_val": overlap_train_val,
    "train_test": overlap_train_test,
    "val_test": overlap_val_test,
    "all_three": overlap_all
})

def show_samples(df, n=5, seed=42):
    cols = ["query", "intent"]
    return df.sample(n=min(n, len(df)), random_state=seed)[cols]

for name, df in datasets.items():
    print("\n==", name, "samples==")
    print(show_samples(df).to_string(index=False))

{'train_val': 0, 'train_test': 0, 'val_test': 0, 'all_three': 0}

== train samples==
                                                                                             query        intent
                                                                               budget alt for this similar_items
                                                                              what pants with this graph_pairing
                                                                              gloves to match this graph_pairing
What should I layer over this spaghetti-strap top? Need something that goes with it but not denim. graph_pairing
                              Any other colorways for the same sneakers? I want them in all-white. color_variant

== val samples==
                                 query        intent
       similar one to wear with straps similar_items
alternative to this with a slit to buy similar_items
         this in a darker shade to get color_variant
       